<a href="https://colab.research.google.com/github/lamphuocgiahy-star/AI-assistant/blob/main/AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "gradio>=5,<6" pypdf python-docx python-pptx

In [ ]:
import os

def get_secret(name, prompt):
    """Đọc khóa từ Colab Secrets (biểu tượng 🔑 bên trái); nếu chưa có thì hỏi trực tiếp."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        import getpass
        value = getpass.getpass(prompt)
    os.environ[name] = value.strip()

get_secret("GEMINI_API_KEY", "Dán Gemini API key: ")
get_secret("SUPABASE_URL", "Dán Supabase Project URL (dạng https://xxxx.supabase.co): ")
get_secret("SUPABASE_KEY", "Dán Supabase publishable key (hoặc anon key): ")
print("✅ Đã nhận các khóa (chỉ lưu trong phiên này)")

Dán Supabase Project URL (dạng https://xxxx.supabase.co): ··········
✅ Đã nhận các khóa (chỉ lưu trong phiên này)


In [ ]:
# ============================================================
# BACKEND: gọi AI (Gemini) + đọc tài liệu + xếp lịch tự học
# Ô này KHÔNG phụ thuộc giao diện, có thể test riêng.
# ============================================================
import os, re, math, requests
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo
import pandas as pd

# ---------- Cấu hình ----------
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")  # đổi tên model ở đây nếu cần
API_URL = "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
MAX_DOC_CHARS = 400_000          # giới hạn độ dài tài liệu gửi cho AI
TZ = ZoneInfo("Asia/Ho_Chi_Minh")

MODES = ["Giải thích khái niệm", "Tóm tắt tài liệu", "Tạo câu hỏi ôn tập"]
LEVELS = ["Dễ hiểu (như mới học lần đầu)", "Vừa phải", "Chuyên sâu"]

MODE_HINT = {
    MODES[0]: "Nhiệm vụ: giải thích khái niệm hoặc câu hỏi của người học. Cấu trúc gợi ý: định nghĩa ngắn → giải thích → ví dụ → lỗi thường gặp (nếu có).",
    MODES[1]: "Nhiệm vụ: tóm tắt tài liệu thành các ý chính có cấu trúc rõ ràng, ý quan trọng nhất đặt trước. Nếu người học nêu phạm vi cụ thể (chương, trang), chỉ tóm tắt phạm vi đó.",
    MODES[2]: "Nhiệm vụ: tạo 5 câu hỏi ôn tập trắc nghiệm (A-D) bám sát tài liệu, từ dễ đến khó. "
    "Chỉ liệt kê câu hỏi, KHÔNG nêu đáp án. Chỉ chấm và giải thích sau khi người học trả lời.",
}
LEVEL_HINT = {
    LEVELS[0]: "Giải thích thật dễ hiểu như cho người mới học lần đầu: ngôn ngữ đời thường, ví dụ gần gũi, giải thích thuật ngữ khi dùng.",
    LEVELS[1]: "Giải thích ở mức vừa phải: dùng thuật ngữ chính kèm định nghĩa ngắn và một ví dụ.",
    LEVELS[2]: "Giải thích chuyên sâu, chính xác: đi vào cơ chế, điều kiện áp dụng, chứng minh nếu phù hợp; giả định người học đã có kiến thức nền.",
}


# ============================================================
# 1. GỌI GEMINI QUA REST API (chỉ cần thư viện requests)
# ============================================================
def _api_key():
    return os.environ.get("GEMINI_API_KEY", "").strip()


def list_models():
    """Liệt kê các model mà API key của bạn dùng được (hữu ích khi gặp lỗi 404)."""
    r = requests.get(
        "https://generativelanguage.googleapis.com/v1beta/models",
        headers={"x-goog-api-key": _api_key()},
        params={"pageSize": 100},
        timeout=30,
    )
    r.raise_for_status()
    return [
        m["name"].replace("models/", "")
        for m in r.json().get("models", [])
        if "generateContent" in m.get("supportedGenerationMethods", [])
    ]


def _friendly_error(resp):
    try:
        detail = resp.json().get("error", {}).get("message", "")
    except Exception:
        detail = resp.text[:200]
    code = resp.status_code
    if code in (400, 401, 403):
        return f"⚠️ API key không hợp lệ hoặc chưa được cấp quyền ({code}). Kiểm tra lại key. Chi tiết: {detail}"
    if code == 404:
        return (f"⚠️ Không tìm thấy model `{GEMINI_MODEL}`. Chạy `list_models()` trong một ô code để xem "
                f"tên model dùng được, rồi đổi biến GEMINI_MODEL. Chi tiết: {detail}")
    if code == 429:
        return "⚠️ Đã chạm giới hạn số lượt gọi AI (gói miễn phí có giới hạn). Đợi khoảng 1 phút rồi thử lại."
    if code >= 500:
        return "⚠️ Máy chủ Gemini đang quá tải hoặc gặp sự cố. Thử lại sau ít phút."
    return f"⚠️ Lỗi {code}: {detail}"


def call_gemini(system_prompt, messages, temperature=0.4):
    """messages: list dạng [{"role": "user"|"assistant", "content": "..."}]. Trả về chuỗi văn bản."""
    if not _api_key():
        return "⚠️ Chưa có API key. Hãy chạy ô nhập API key ở phía trên."
    contents = [
        {"role": "user" if m["role"] == "user" else "model", "parts": [{"text": m["content"]}]}
        for m in messages
    ]
    body = {
        "system_instruction": {"parts": [{"text": system_prompt}]},
        "contents": contents,
        "generationConfig": {"temperature": temperature},
    }
    try:
        resp = requests.post(
            API_URL.format(model=GEMINI_MODEL),
            headers={"x-goog-api-key": _api_key(), "Content-Type": "application/json"},
            json=body,
            timeout=120,
        )
    except requests.RequestException as e:
        return f"⚠️ Không kết nối được tới Gemini: {e}"
    if resp.status_code != 200:
        return _friendly_error(resp)
    data = resp.json()
    try:
        parts = data["candidates"][0]["content"]["parts"]
        text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
        return text.strip() or "⚠️ AI không trả về nội dung. Hãy thử hỏi lại."
    except (KeyError, IndexError):
        reason = data.get("promptFeedback", {}).get("blockReason", "không rõ")
        return f"⚠️ AI không thể trả lời câu hỏi này (lý do: {reason}). Hãy thử diễn đạt lại."


# ============================================================
# 2. ĐỌC TÀI LIỆU (PDF / DOCX / PPTX / TXT)
# ============================================================
_doc_cache = {}


def read_document(path):
    stamp = (path, os.path.getmtime(path))
    if stamp in _doc_cache:
        return _doc_cache[stamp]
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        pages = []
        for i, page in enumerate(PdfReader(path).pages, 1):
            t = (page.extract_text() or "").strip()
            if t:
                pages.append(f"[Trang {i}]\n{t}")
        text = "\n\n".join(pages)
    elif ext == ".docx":
        import docx
        d = docx.Document(path)
        lines = [p.text for p in d.paragraphs if p.text.strip()]
        for table in d.tables:
            for row in table.rows:
                lines.append(" | ".join(c.text.strip() for c in row.cells))
        text = "\n".join(lines)
    elif ext == ".pptx":
        from pptx import Presentation
        slides = []
        for i, slide in enumerate(Presentation(path).slides, 1):
            texts = [sh.text_frame.text for sh in slide.shapes
                     if sh.has_text_frame and sh.text_frame.text.strip()]
            if texts:
                slides.append(f"[Slide {i}]\n" + "\n".join(texts))
        text = "\n\n".join(slides)
    else:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
    _doc_cache[stamp] = text
    return text


def build_context(files):
    """Gộp nội dung các file đã tải lên. Trả về (văn bản, danh sách cảnh báo)."""
    if not files:
        return "", []
    notes, chunks, total = [], [], 0
    for f in files:
        path = f if isinstance(f, str) else f.name
        name = os.path.basename(path)
        try:
            text = read_document(path)
        except Exception as e:
            notes.append(f"Không đọc được '{name}': {e}")
            continue
        if not text.strip():
            notes.append(f"'{name}' không có chữ đọc được (có thể là bản scan/ảnh).")
            continue
        room = MAX_DOC_CHARS - total
        if room <= 0:
            notes.append(f"Bỏ qua '{name}' vì tổng tài liệu quá dài.")
            continue
        if len(text) > room:
            text = text[:room]
            notes.append(f"'{name}' quá dài nên chỉ dùng phần đầu.")
        total += len(text)
        chunks.append(f"=== TÀI LIỆU: {name} ===\n{text}")
    return "\n\n".join(chunks), notes

PEDAGOGY_RULES = """
Bạn là gia sư hướng dẫn học tập.
Cần tuân theo phương pháp sư phạm giúp hỗ trợ việc ghi nhớ và thấu hiểu khái niệm.
Các quy tắc bạn cần tuần theo như sau:

1. Khi người dùng đưa ra câu hỏi về 1 khái niệm chung, hãy trả lời ngắn gọn, đúng trọng tâm rồi đặt ra câu hỏi ngược để kích thích người đọc suy nghĩ.
2. Khi người dùng nhờ giải quyết bài tập hộ, không đưa ra đáp án, phải đưa ra gợi ý liên quan rồi hướng dẫn người dùng giải quyết. Hãy tăng sự trợ giúp cho người đọc nếu họ liên tục gặp khó, bắt đầu từ (1) là đưa ra gợi ý, (2) là đưa ra ví dụ, (3) là đưa ra giải thích và chỉ tiết lộ đáp án đúng khi người dùng đưa ra 1 đáp án gần đúng hoặc hợp lý.
3. Khi người dùng trả lời sai, hãy hỏi cách họ suy nghĩ, để hiểu rằng họ đang gặp vấn đề gì: thiếu thông tin cần thiết, hiểu nhầm hoặc đang nhìn vấn đề sai cách.
4. Các câu trả lời đưa ra ngắn gọn, hợp lý, không giảng đạo, thân thiện với người dùng.
"""

def build_system_prompt(mode, level, context):
    doc_part = (
        "--- TÀI LIỆU CỦA NGƯỜI HỌC ---\n" + context
        if context else "Người học chưa tải tài liệu nào."
    )
    return (
        "Bạn là trợ lý học tập cho sinh viên Việt Nam. Luôn trả lời bằng tiếng Việt "
        "(giữ thuật ngữ tiếng Anh khi cần, kèm giải thích).\n"
        f"{MODE_HINT[mode]}\n{LEVEL_HINT[level]}\n"
        f"{PEDAGOGY_RULES}"
        "Quy tắc:\n"
        "- Khi có tài liệu: ưu tiên dựa vào tài liệu và ghi nguồn dạng (Trang N) hoặc (Slide N) nếu có. "
        "Nếu tài liệu không đề cập, nói rõ 'Tài liệu không đề cập', rồi mới bổ sung kiến thức chung "
        "và nói rõ đó là kiến thức ngoài tài liệu.\n"
        "- Không bịa số liệu, trích dẫn hay công thức. Nếu không chắc, hãy nói không chắc.\n"
        "- Trình bày ngắn gọn, dùng Markdown. Công thức viết dạng văn bản dễ đọc (ví dụ x^2).\n\n"
        f"{doc_part}"
        "Nhắc lại: luôn tuân theo phần PHƯƠNG PHÁP SƯ PHẠM ở trên."
    )


# ============================================================
# 3. XẾP LỊCH TỰ HỌC (thuật toán thường, không dùng AI)
#    Lý do: xếp lịch cần chính xác tuyệt đối, AI dễ nhầm giờ.
# ============================================================
DAY_NAMES = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
WINDOWS = {"Sáng": (6 * 60, 12 * 60), "Chiều": (12 * 60, 18 * 60), "Tối": (18 * 60, 22 * 60 + 30)}
MAX_SESSION, MIN_SESSION, BUFFER = 90, 30, 15   # phút


def _s(x):
    return "" if x is None or (isinstance(x, float) and math.isnan(x)) else str(x).strip()


def clean_table(df, headers):
    """Bỏ các dòng trống (cột đầu tiên rỗng) trong bảng người dùng nhập."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=headers)
    keep = df.iloc[:, 0].map(_s) != ""
    return df[keep].reset_index(drop=True)


def parse_weekday(s):
    s = _s(s).lower()
    if not s:
        return None
    if "cn" in s or "chủ" in s:
        return 6
    m = re.search(r"\d", s)
    if m and 2 <= int(m.group()) <= 7:
        return int(m.group()) - 2
    return None


def parse_time(s):
    m = re.match(r"^\s*(\d{1,2})\s*[:hH]\s*(\d{0,2})\s*$", _s(s))
    if not m:
        return None
    h, mi = int(m.group(1)), int(m.group(2) or 0)
    return h * 60 + mi if h < 24 and mi < 60 else None


def parse_due(s, today):
    m = re.match(r"^\s*(\d{1,2})\s*[/\-.]\s*(\d{1,2})(?:\s*[/\-.]\s*(\d{2,4}))?\s*$", _s(s))
    if not m:
        return None
    d, mo = int(m.group(1)), int(m.group(2))
    y = int(m.group(3)) if m.group(3) else today.year
    if y < 100:
        y += 2000
    try:
        due = date(y, mo, d)
        if not m.group(3) and due < today and (today - due).days > 60:
            due = date(y + 1, mo, d)   # nhập "10/01" vào tháng 10 -> hiểu là năm sau
    except ValueError:
        return None
    return due


def parse_hours(s):
    try:
        h = float(_s(s).replace(",", "."))
    except ValueError:
        return None
    return h if h > 0 else None


def free_blocks(day_classes, earliest):
    """Các khoảng trống trong ngày (đã trừ giờ học trên trường), chia theo Sáng/Chiều/Tối."""
    lo, hi = WINDOWS["Sáng"][0], WINDOWS["Tối"][1]
    cursor, free = max(lo, earliest), []
    for s, e in sorted((s - BUFFER, e + BUFFER) for s, e in day_classes):
        if s > cursor:
            free.append((cursor, min(s, hi)))
        cursor = max(cursor, e)
    if cursor < hi:
        free.append((cursor, hi))
    blocks = []
    for a, b in free:
        for name, (ws, we) in WINDOWS.items():
            s, e = max(a, ws), min(b, we)
            if e - s >= MIN_SESSION:
                blocks.append([s, e, name])
    return blocks


def take_time(blocks, minutes, preferred):
    """Lấy tối đa `minutes` phút từ các khoảng trống, ưu tiên khung giờ ưa thích."""
    blocks.sort(key=lambda b: (0 if b[2] == preferred else 1, b[0]))
    got = []
    for b in blocks:
        while minutes >= MIN_SESSION and b[1] - b[0] >= MIN_SESSION:
            length = min(MAX_SESSION, minutes, b[1] - b[0])
            got.append((b[0], b[0] + length))
            b[0] += length
            minutes -= length
        if minutes < MIN_SESSION:
            break
    return got


def build_schedule(timetable_df, deadlines_df, hours_per_day, preferred, now=None):
    """Trả về (bảng lịch DataFrame, danh sách cảnh báo)."""
    now = now or datetime.now(TZ)
    today = now.date()
    problems = []

    # Giờ học trên trường theo từng thứ
    classes = {}
    for _, r in clean_table(timetable_df, ["Môn học"] * 5).iterrows():
        wd, s, e = parse_weekday(r.iloc[1]), parse_time(r.iloc[2]), parse_time(r.iloc[3])
        if wd is None or s is None or e is None or s >= e:
            problems.append(f"Bỏ qua dòng thời khóa biểu '{_s(r.iloc[0])}': thứ/giờ không đọc được (ví dụ: 2, 07:00, 09:30).")
            continue
        classes.setdefault(wd, []).append((s, e))

    # Danh sách việc cần làm
    tasks = []
    for _, r in clean_table(deadlines_df, ["Công việc"] * 4).iterrows():
        name, subject = _s(r.iloc[0]), _s(r.iloc[1])
        due, hrs = parse_due(r.iloc[2], today), parse_hours(r.iloc[3])
        if due is None:
            problems.append(f"Bỏ qua '{name}': hạn không đọc được (nhập dạng dd/mm).")
        elif hrs is None:
            problems.append(f"Bỏ qua '{name}': số giờ không hợp lệ.")
        elif due < today:
            problems.append(f"Bỏ qua '{name}': hạn đã qua.")
        else:
            tasks.append({"name": name + (f" ({subject})" if subject else ""),
                          "due": due, "left": int(math.ceil(hrs * 2)) * 30})
    if not tasks:
        return pd.DataFrame(columns=["Ngày", "Khung giờ", "Nội dung"]), problems

    last_day = min(max(t["due"] for t in tasks), today + timedelta(days=90))
    rows, d = [], today
    while d <= last_day and any(t["left"] > 0 for t in tasks):
        earliest = 0
        if d == today:
            earliest = ((now.hour * 60 + now.minute + 30 + 29) // 30) * 30
        blocks = free_blocks(classes.get(d.weekday(), []), earliest)
        capacity = int(hours_per_day * 60)
        active = sorted((t for t in tasks if t["left"] > 0 and t["due"] >= d), key=lambda t: t["due"])
        for t in active:
            if capacity < MIN_SESSION:
                break
            days_left = max(1, (t["due"] - d).days)       # chia đều khối lượng cho các ngày còn lại
            quota = int(math.ceil(t["left"] / days_left / 30)) * 30
            got = take_time(blocks, min(quota, t["left"], capacity), preferred)
            used = sum(e - s for s, e in got)
            t["left"] -= used
            capacity -= used
            rows += [(d, s, e, t["name"]) for s, e in got]
        d += timedelta(days=1)

    for t in tasks:
        if t["left"] > 0:
            problems.append(f"Chưa đủ thời gian cho '{t['name']}': còn thiếu {t['left'] / 60:g} giờ trước hạn "
                            f"{t['due']:%d/%m}. Hãy tăng số giờ tự học mỗi ngày hoặc giảm khối lượng.")
    rows.sort()
    table = pd.DataFrame(
        [[f"{DAY_NAMES[d.weekday()]} {d:%d/%m}",
          f"{s // 60:02d}:{s % 60:02d}–{e // 60:02d}:{e % 60:02d}", name] for d, s, e, name in rows],
        columns=["Ngày", "Khung giờ", "Nội dung"],
    )
    return table, problems


In [ ]:
# ============================================================
# TẦNG DỮ LIỆU: đăng ký, đăng nhập, lưu và tải dữ liệu qua Supabase
# Dùng thư viện requests gọi thẳng API (không cần cài thêm gói nào).
# Ô này KHÔNG phụ thuộc giao diện.
# ============================================================
MAX_TT_ROWS = 30      # phải khớp với số trong file SQL
MAX_TASKS = 50
MIN_PASSWORD = 8

TT_KEYS = ["subject", "weekday", "start_time", "end_time", "room"]
TASK_KEYS = ["title", "subject", "due", "hours"]


def _url():
    return os.environ.get("SUPABASE_URL", "").strip().rstrip("/")


def _key():
    return os.environ.get("SUPABASE_KEY", "").strip()


def _headers(token=None):
    h = {"apikey": _key(), "Content-Type": "application/json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def _configured():
    return bool(_url() and _key())


def _msg_from(resp):
    try:
        d = resp.json()
        return str(d.get("msg") or d.get("error_description") or d.get("message") or d.get("error") or "")
    except Exception:
        return resp.text[:200]


def _auth_error(resp):
    msg = _msg_from(resp)
    low = msg.lower()
    if resp.status_code == 429 or "rate limit" in low:
        return "Gửi quá nhiều yêu cầu, hãy thử lại sau ít phút."
    if "invalid login" in low:
        return "Email hoặc mật khẩu không đúng."
    if "not confirmed" in low:
        return "Bạn cần bấm liên kết xác nhận trong email trước khi đăng nhập."
    if "already" in low and "registered" in low:
        return "Email này đã được đăng ký."
    if "password" in low and ("short" in low or "least" in low or "weak" in low):
        return "Mật khẩu chưa đạt yêu cầu (quá ngắn hoặc quá yếu)."
    return f"Lỗi {resp.status_code}: {msg}"


def _data_error(resp):
    msg = _msg_from(resp)
    text = resp.text.lower()
    if resp.status_code == 401 or "jwt expired" in text:
        return "Phiên đăng nhập đã hết hạn, hãy đăng nhập lại."
    if "pgrst205" in text or "pgrst202" in text or "schema cache" in text:
        return "Chưa thiết lập cơ sở dữ liệu: hãy chạy file SQL trong Supabase (SQL Editor)."
    if resp.status_code == 400 and msg:
        return f"Không thực hiện được: {msg}"
    return f"Lỗi {resp.status_code}: {msg}"


def _check_credentials(email, password):
    if not _configured():
        return "Chưa cấu hình Supabase (SUPABASE_URL và SUPABASE_KEY). Hãy chạy ô nhập khóa."
    if not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", (email or "").strip()):
        return "Email không hợp lệ."
    if len(password or "") < MIN_PASSWORD:
        return f"Mật khẩu cần ít nhất {MIN_PASSWORD} ký tự."
    return ""


def sb_sign_up(email, password):
    """Trả về (thành công?, thông báo)."""
    err = _check_credentials(email, password)
    if err:
        return False, err
    try:
        r = requests.post(f"{_url()}/auth/v1/signup", headers=_headers(),
                          json={"email": email.strip(), "password": password}, timeout=30)
    except requests.RequestException as e:
        return False, f"Không kết nối được tới Supabase: {e}"
    if r.status_code not in (200, 201):
        return False, _auth_error(r)
    if r.json().get("access_token"):
        return True, "Đăng ký thành công. Bây giờ hãy bấm Đăng nhập."
    return True, "Đăng ký thành công. Hãy kiểm tra email để xác nhận, rồi bấm Đăng nhập."


def sb_sign_in(email, password):
    """Trả về (thành công?, thông báo, phiên). Phiên chỉ giữ thứ tối thiểu cần dùng."""
    err = _check_credentials(email, password)
    if err:
        return False, err, None
    try:
        r = requests.post(f"{_url()}/auth/v1/token", params={"grant_type": "password"},
                          headers=_headers(), json={"email": email.strip(), "password": password}, timeout=30)
    except requests.RequestException as e:
        return False, f"Không kết nối được tới Supabase: {e}", None
    if r.status_code != 200:
        return False, _auth_error(r), None
    d = r.json()
    session = {
        "access_token": d["access_token"],
        "user_id": d["user"]["id"],
        "email": d["user"].get("email", email.strip()),
    }
    return True, "Đăng nhập thành công.", session


def sb_load(session):
    """Tải dữ liệu của người đang đăng nhập. Trả về (hàng_tkb, hàng_việc, lỗi)."""
    try:
        results = []
        for table, keys in (("timetable_entries", TT_KEYS), ("tasks", TASK_KEYS)):
            r = requests.get(f"{_url()}/rest/v1/{table}", headers=_headers(session["access_token"]),
                             params={"select": ",".join(keys), "order": "sort_order.asc"}, timeout=30)
            if r.status_code != 200:
                return None, None, _data_error(r)
            results.append([[_s(row.get(k)) for k in keys] for row in r.json()])
        return results[0], results[1], ""
    except requests.RequestException as e:
        return None, None, f"Không kết nối được tới Supabase: {e}"


def sb_save(session, tt_rows, task_rows):
    """Lưu toàn bộ dữ liệu trong một giao dịch. Trả về chuỗi lỗi ('' nếu thành công)."""
    if len(tt_rows) > MAX_TT_ROWS or len(task_rows) > MAX_TASKS:
        return f"Vượt giới hạn: tối đa {MAX_TT_ROWS} dòng thời khóa biểu và {MAX_TASKS} việc cần làm."
    payload = {
        "p_tt": [dict(zip(TT_KEYS, row)) for row in tt_rows],
        "p_tasks": [dict(zip(TASK_KEYS, row)) for row in task_rows],
    }
    try:
        r = requests.post(f"{_url()}/rest/v1/rpc/save_my_data",
                          headers=_headers(session["access_token"]), json=payload, timeout=30)
    except requests.RequestException as e:
        return f"Không kết nối được tới Supabase: {e}"
    return "" if r.status_code in (200, 204) else _data_error(r)


def sb_delete_account(session):
    """Xóa tài khoản và toàn bộ dữ liệu. Trả về chuỗi lỗi ('' nếu thành công)."""
    try:
        r = requests.post(f"{_url()}/rest/v1/rpc/delete_my_account",
                          headers=_headers(session["access_token"]), json={}, timeout=30)
    except requests.RequestException as e:
        return f"Không kết nối được tới Supabase: {e}"
    return "" if r.status_code in (200, 204) else _data_error(r)

In [ ]:
print(call_gemini("Bạn là trợ lý.", [{"role": "user", "content": "Chào bạn, 1+1 bằng mấy?"}]))
print("Supabase đã cấu hình:", _configured())

Chào bạn, 1+1 bằng 2 nhé! Mình có thể giúp gì thêm cho bạn không?
Supabase đã cấu hình: True


In [ ]:
# ============================================================
# GIAO DIỆN (Gradio) - có tài khoản và lưu dữ liệu
# ============================================================
import gradio as gr
import pandas as pd

TIMETABLE_HEADERS = ["Môn học", "Thứ", "Giờ bắt đầu", "Giờ kết thúc", "Phòng"]
DEADLINE_HEADERS = ["Công việc", "Môn học", "Hạn (dd/mm)", "Số giờ ước tính"]

EXAMPLE_TIMETABLE = [
    ["Giải tích 1", "2", "07:00", "09:30", "A101"],
    ["Lập trình căn bản", "4", "13:00", "16:00", "B203"],
    ["", "", "", "", ""],
]
EXAMPLE_DEADLINES = [
    ["Bài tập lớn Lập trình", "Lập trình căn bản", "30/10", "6"],
    ["", "", "", ""],
]
EMPTY_SCHEDULE = pd.DataFrame(columns=["Ngày", "Khung giờ", "Nội dung"])


def status_text(session):
    if session:
        return f"✅ Đã đăng nhập: **{session['email']}**"
    return "⚪ Bạn chưa đăng nhập. Vẫn dùng được, nhưng dữ liệu sẽ không được lưu."


def to_df(rows, headers):
    """Thêm một dòng trống ở cuối để người dùng gõ tiếp."""
    return pd.DataFrame(rows + [[""] * len(headers)], columns=headers)


def table_rows(df, headers):
    """Bảng người dùng nhập -> danh sách hàng (chuỗi), bỏ dòng trống."""
    return [[_s(c) for c in row] for row in clean_table(df, headers).values.tolist()]


# ---------- Tính năng 1: Học tập ----------
def chat_fn(message, history, mode, level, files):
    history = history or []
    if not message or not message.strip():
        return history, ""

    context, notes = build_context(files)
    for n in notes:
        gr.Warning(n)

    if mode in (MODES[1], MODES[2]) and not context:
        reply = "Bạn hãy **tải tài liệu lên** ở khung bên trái trước nhé, chế độ này cần có tài liệu để làm việc."
    else:
        system = build_system_prompt(mode, level, context)
        recent = [
            {"role": m["role"], "content": m["content"]}
            for m in history[-10:]
            if isinstance(m.get("content"), str)
        ]
        while recent and recent[0]["role"] != "user":
            recent.pop(0)
        reply = call_gemini(system, recent + [{"role": "user", "content": message}])

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ]
    return history, ""


def add_to_plan(topic, due, hours, deadlines_df):
    if not topic or not topic.strip():
        gr.Warning("Hãy nhập chủ đề cần ôn tập.")
        return deadlines_df
    rows = table_rows(deadlines_df, DEADLINE_HEADERS)
    rows.append([f"Ôn tập: {topic.strip()}", "", due or "", str(hours)])
    gr.Info("Đã thêm vào danh sách việc ở tab 📅 Lịch học. Nhớ bấm Lưu nếu muốn giữ lại.")
    return pd.DataFrame(rows, columns=DEADLINE_HEADERS)


# ---------- Tính năng 2: Lên lịch ----------
def make_schedule(timetable_df, deadlines_df, hours_per_day, preferred_time):
    table, problems = build_schedule(timetable_df, deadlines_df, hours_per_day, preferred_time)
    for p in problems:
        gr.Warning(p)
    if len(table) == 0 and not problems:
        gr.Warning("Chưa có công việc nào để lên lịch.")
    return table


# ---------- Tài khoản và lưu dữ liệu ----------
def do_sign_up(email, password):
    ok, msg = sb_sign_up(email, password)
    (gr.Info if ok else gr.Warning)(msg)


def do_sign_in(email, password, session):
    ok, msg, new_session = sb_sign_in(email, password)
    if not ok:
        gr.Warning(msg)
        return session, status_text(session), gr.update(), gr.update(), gr.update()
    tt, tasks, err = sb_load(new_session)
    if err:
        gr.Warning(err)
        return new_session, status_text(new_session), gr.update(), gr.update(), ""
    gr.Info("Đăng nhập thành công." + (" Đã tải dữ liệu của bạn." if (tt or tasks) else ""))
    return (
        new_session,
        status_text(new_session),
        to_df(tt, TIMETABLE_HEADERS) if tt else gr.update(),
        to_df(tasks, DEADLINE_HEADERS) if tasks else gr.update(),
        "",   # xóa ô mật khẩu
    )


def _logged_out_view():
    """Trạng thái sạch: tránh để dữ liệu người trước hiện ra trên máy dùng chung."""
    return (None, status_text(None), EXAMPLE_TIMETABLE, EXAMPLE_DEADLINES, [], EMPTY_SCHEDULE)


def do_sign_out():
    gr.Info("Đã đăng xuất.")
    return _logged_out_view()


def do_save(session, timetable_df, deadlines_df):
    if not session:
        gr.Warning("Hãy đăng nhập ở tab 👤 Tài khoản để lưu dữ liệu.")
        return
    tt = table_rows(timetable_df, TIMETABLE_HEADERS)
    tasks = table_rows(deadlines_df, DEADLINE_HEADERS)
    err = sb_save(session, tt, tasks)
    if err:
        gr.Warning(err)
    else:
        gr.Info(f"Đã lưu {len(tt)} dòng thời khóa biểu và {len(tasks)} việc cần làm.")


def do_reload(session):
    if not session:
        gr.Warning("Hãy đăng nhập ở tab 👤 Tài khoản trước.")
        return gr.update(), gr.update()
    tt, tasks, err = sb_load(session)
    if err:
        gr.Warning(err)
        return gr.update(), gr.update()
    gr.Info("Đã tải lại dữ liệu đã lưu.")
    return to_df(tt, TIMETABLE_HEADERS), to_df(tasks, DEADLINE_HEADERS)


def do_delete(session, confirmed):
    keep = (session, status_text(session), gr.update(), gr.update(), gr.update(), gr.update())
    if not session:
        gr.Warning("Bạn chưa đăng nhập.")
        return keep
    if not confirmed:
        gr.Warning("Hãy tích vào ô xác nhận trước khi xóa.")
        return keep
    err = sb_delete_account(session)
    if err:
        gr.Warning(err)
        return keep
    gr.Info("Đã xóa tài khoản và toàn bộ dữ liệu của bạn.")
    return _logged_out_view()


# ---------- Giao diện ----------
CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
#title {text-align: center; margin-bottom: 0;}
#subtitle {text-align: center; color: #666; margin-top: 0;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="Trợ lý sinh viên") as demo:
    session = gr.State(None)   # thông tin đăng nhập, riêng cho từng trình duyệt

    gr.Markdown("# 🎓 Trợ lý sinh viên", elem_id="title")
    gr.Markdown("Hiểu bài nhanh hơn và sắp xếp việc học hợp lý hơn.", elem_id="subtitle")

    with gr.Tabs():
        # ===== TAB 1: HỌC TẬP =====
        with gr.Tab("📚 Học tập"):
            with gr.Row():
                with gr.Column(scale=1):
                    files = gr.File(
                        label="Tài liệu học (PDF, Word, PowerPoint, TXT)",
                        file_count="multiple",
                        file_types=[".pdf", ".docx", ".txt", ".pptx"],
                    )
                    mode = gr.Radio(MODES, value=MODES[0], label="Bạn muốn AI làm gì?")
                    level = gr.Dropdown(LEVELS, value=LEVELS[0], label="Mức độ giải thích")
                with gr.Column(scale=2):
                    chatbot = gr.Chatbot(
                        type="messages", height=420, label="Trò chuyện", show_copy_button=True,
                        placeholder="Tải tài liệu lên rồi đặt câu hỏi, ví dụ: 'Đạo hàm riêng là gì?'",
                    )
                    with gr.Row():
                        msg = gr.Textbox(
                            placeholder="Nhập câu hỏi hoặc khái niệm cần giải thích...",
                            show_label=False, scale=5, container=False,
                        )
                        send = gr.Button("Gửi", variant="primary", scale=1)
                    clear = gr.Button("Xóa cuộc trò chuyện", size="sm")

            with gr.Accordion("📌 Đưa chủ đề này vào lịch ôn tập", open=False):
                with gr.Row():
                    plan_topic = gr.Textbox(label="Chủ đề cần ôn", scale=3)
                    plan_due = gr.Textbox(label="Hạn / ngày thi (dd/mm)", scale=2)
                    plan_hours = gr.Number(label="Số giờ dự kiến", value=2, minimum=0.5, scale=1)
                add_btn = gr.Button("Thêm vào danh sách việc cần lên lịch")

        # ===== TAB 2: LỊCH HỌC =====
        with gr.Tab("📅 Lịch học"):
            gr.Markdown("### 1. Thời khóa biểu trên trường")
            timetable = gr.Dataframe(
                headers=TIMETABLE_HEADERS, value=EXAMPLE_TIMETABLE,
                datatype=["str"] * 5, row_count=(3, "dynamic"), col_count=(5, "fixed"),
                interactive=True,
                label="Thứ: 2-7 hoặc CN. Giờ dạng 07:00. Có thể dán từ bảng tính.",
            )
            gr.Markdown("### 2. Việc cần làm / deadline")
            deadlines = gr.Dataframe(
                headers=DEADLINE_HEADERS, value=EXAMPLE_DEADLINES,
                datatype=["str"] * 4, row_count=(2, "dynamic"), col_count=(4, "fixed"),
                interactive=True,
                label="Các chủ đề từ tab Học tập sẽ được thêm vào đây",
            )
            with gr.Row():
                save_btn = gr.Button("💾 Lưu thời khóa biểu và việc cần làm")
                reload_btn = gr.Button("🔄 Tải lại dữ liệu đã lưu")
            gr.Markdown("*Cần đăng nhập ở tab 👤 Tài khoản để lưu.*")

            gr.Markdown("### 3. Thói quen tự học")
            with gr.Row():
                hours_per_day = gr.Slider(1, 8, value=3, step=0.5, label="Số giờ tự học mỗi ngày")
                preferred_time = gr.Radio(["Sáng", "Chiều", "Tối"], value="Tối", label="Khung giờ ưa thích")
            gen_btn = gr.Button("✨ Tạo lịch tự học", variant="primary")
            schedule_out = gr.Dataframe(label="Lịch gợi ý", interactive=False)

        # ===== TAB 3: TÀI KHOẢN =====
        with gr.Tab("👤 Tài khoản"):
            status = gr.Markdown(status_text(None))
            email = gr.Textbox(label="Email")
            password = gr.Textbox(label=f"Mật khẩu (tối thiểu {MIN_PASSWORD} ký tự)", type="password")
            with gr.Row():
                signup_btn = gr.Button("Đăng ký")
                signin_btn = gr.Button("Đăng nhập", variant="primary")
                signout_btn = gr.Button("Đăng xuất")
            with gr.Accordion("⚠️ Xóa tài khoản và toàn bộ dữ liệu", open=False):
                confirm = gr.Checkbox(label="Tôi hiểu việc này không thể hoàn tác")
                delete_btn = gr.Button("Xóa vĩnh viễn", variant="stop")
            gr.Markdown(
                "**Quyền riêng tư:** tài liệu bạn tải lên không được lưu. Chỉ thời khóa biểu và "
                "việc cần làm được lưu. Câu hỏi và nội dung tài liệu được gửi cho Google Gemini để xử lý."
            )

    # ----- Gắn sự kiện -----
    chat_inputs = [msg, chatbot, mode, level, files]
    send.click(chat_fn, chat_inputs, [chatbot, msg])
    msg.submit(chat_fn, chat_inputs, [chatbot, msg])
    clear.click(lambda: [], None, chatbot)
    add_btn.click(add_to_plan, [plan_topic, plan_due, plan_hours, deadlines], deadlines)
    gen_btn.click(make_schedule, [timetable, deadlines, hours_per_day, preferred_time], schedule_out)

    signup_btn.click(do_sign_up, [email, password], None)
    signin_btn.click(do_sign_in, [email, password, session], [session, status, timetable, deadlines, password])
    full_reset = [session, status, timetable, deadlines, chatbot, schedule_out]
    signout_btn.click(do_sign_out, None, full_reset)
    delete_btn.click(do_delete, [session, confirm], full_reset)
    save_btn.click(do_save, [session, timetable, deadlines], None)
    reload_btn.click(do_reload, [session], [timetable, deadlines])

demo.launch(share=True, debug=True)

/tmp/ipykernel_7487/4184979641.py:171: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="Trợ lý sinh viên") as demo:
/tmp/ipykernel_7487/4184979641.py:171: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="Trợ lý sinh viên") as demo:
/tmp/ipykernel_7487/4184979641.py:190: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_7487/4184979641.py:190: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chat

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://556d6c9b5d56475b77.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
